# Milestone 3 – CNN on SVHN (Code Smells – Individually Isolated)

Baseline CNN with **4 ML code smells**, each demonstrated in isolation:

| Section | Active Smell |
|---|---|
| 1 | Unbounded Graph Expansion |
| 2 | Graph-Constant Bottleneck |
| 3 | GPU Released Memory Failure |
| 4 | Shape Mismatch Leak |

Each section runs 10 independent training runs. Only one smell is active per section; the other three are absent.

In [ ]:
!pip install -q codecarbon scipy

In [ ]:
# Fix codecarbon missing nordic_emissions.json
import codecarbon, os, json

_data_dir = os.path.join(os.path.dirname(codecarbon.__file__), 'data', 'private_infra')
os.makedirs(_data_dir, exist_ok=True)
_nordic_path = os.path.join(_data_dir, 'nordic_emissions.json')
if not os.path.exists(_nordic_path):
    with open(_nordic_path, 'w') as _f:
        json.dump({}, _f)
print('CodeCarbon data file ready:', _nordic_path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc, os, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Dropout, BatchNormalization, Flatten
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K

from sklearn.metrics import classification_report, confusion_matrix
from codecarbon import EmissionsTracker

In [ ]:
import tensorflow_datasets as tfds

print('Downloading SVHN dataset via tensorflow_datasets...')
(ds_train, ds_test), ds_info = tfds.load(
    'svhn_cropped',
    split=['train', 'test'],
    shuffle_files=False,
    as_supervised=True,
    with_info=True
)

def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

ds_train = ds_train.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
ds_test  = ds_test.map(preprocess,  num_parallel_calls=tf.data.AUTOTUNE)

X_train = np.stack([img.numpy() for img, _ in ds_train])
y_train = np.array([lbl.numpy() for _, lbl in ds_train])
X_test  = np.stack([img.numpy() for img, _ in ds_test])
y_test  = np.array([lbl.numpy() for _, lbl in ds_test])

y_cat_train = to_categorical(y_train, 10)
y_cat_test  = to_categorical(y_test,  10)

print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}  y_test: {y_test.shape}')

In [ ]:
INPUT_SHAPE = (32, 32, 3)
KERNEL_SIZE = (3, 3)

def build_cnn():
    METRICS = [
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
    model = Sequential()

    # Block 1 – 32 filters
    model.add(Conv2D(filters=32, kernel_size=KERNEL_SIZE, input_shape=INPUT_SHAPE, activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(Conv2D(filters=32, kernel_size=KERNEL_SIZE, activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPool2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # Block 2 – 64 filters
    model.add(Conv2D(filters=64, kernel_size=KERNEL_SIZE, activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(Conv2D(filters=64, kernel_size=KERNEL_SIZE, activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPool2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # Block 3 – 128 filters
    model.add(Conv2D(filters=128, kernel_size=KERNEL_SIZE, activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(Conv2D(filters=128, kernel_size=KERNEL_SIZE, activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPool2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # Fully connected
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.25))
    model.add(Dense(10, activation='softmax'))

    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=METRICS)
    return model

def print_results(results, label):
    print('\n' + '='*80)
    print(f'RESULTS – {label}')
    print('='*80)
    header = f"{'Run':<5} {'EpochStopped':<14} {'Accuracy':<12} {'Precision':<12} {'Recall':<10} {'Energy(kWh)':<14} {'CO2(kg)':<12}"
    print(header)
    print('-'*80)
    for r in results:
        print(f"{r['run']:<5} {r['epoch_stopped']:<14} {r['test_accuracy']:<12.4f} "
              f"{r['test_precision']:<12.4f} {r['test_recall']:<10.4f} "
              f"{r['energy_kwh']:<14.6f} {r['co2_kg']:<12.6f}")
    print('-'*80)
    print(f"{'AVG':<5} {np.mean([r['epoch_stopped'] for r in results]):<14.1f}"
          f" {np.mean([r['test_accuracy'] for r in results]):<12.4f}"
          f" {np.mean([r['test_precision'] for r in results]):<12.4f}"
          f" {np.mean([r['test_recall'] for r in results]):<10.4f}"
          f" {np.mean([r['energy_kwh'] for r in results]):<14.6f}"
          f" {np.mean([r['co2_kg'] for r in results]):<12.6f}")
    print('='*80)

---
## Section 1 – SMELL 1: Unbounded Graph Expansion

**Active smell:** `K.clear_session()` is deliberately omitted before each `build_cnn()` call, so every new model appends nodes to the same growing TensorFlow graph across all 10 runs.

**Absent smells:** No graph-constant bottleneck, no GPU memory leak, no shape mismatch. Data is used directly in its original float32 form; the model is deleted and `gc.collect()` is called after every run.

In [ ]:
# ============================================================
# SMELL 1 – Unbounded Graph Expansion (isolated)
# ============================================================
os.makedirs('emissions', exist_ok=True)
results_smell1 = []

for run in range(1, 11):
    print(f"\n{'='*60}")
    print(f'  [Smell 1] Run {run}/10')
    print(f"{'='*60}")

    # SMELL 1 ACTIVE – K.clear_session() intentionally omitted.
    # Each call to build_cnn() appends new graph nodes to the same
    # default TensorFlow graph; over 10 runs the graph grows without bound.
    # K.clear_session()   <-- intentionally removed

    tracker = EmissionsTracker(
        project_name=f'CNN_SVHN_Smell1_Run{run}',
        output_dir='./emissions',
        save_to_file=True,
        log_level='warning'
    )
    tracker.start()

    model      = build_cnn()
    early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

    # No shape mismatch smell – use original float32 arrays directly
    history = model.fit(
        X_train, y_cat_train,
        epochs=50, batch_size=32,
        validation_data=(X_test, y_cat_test),
        callbacks=[early_stop],
        verbose=1
    )

    co2_kg     = tracker.stop()
    energy_kwh = tracker.final_emissions_data.energy_consumed

    epoch_stopped = len(history.history['loss'])
    eval_res      = model.evaluate(X_test, y_cat_test, verbose=0)

    results_smell1.append(dict(
        run=run,
        epoch_stopped=epoch_stopped,
        test_loss=eval_res[0],
        test_accuracy=eval_res[1],
        test_precision=eval_res[2],
        test_recall=eval_res[3],
        energy_kwh=energy_kwh,
        co2_kg=co2_kg
    ))
    print(f'Run {run} | Epoch stopped: {epoch_stopped} | Acc: {eval_res[1]:.4f} | '
          f'Energy: {energy_kwh:.6f} kWh | CO2: {co2_kg:.6f} kg')

    # No GPU memory smell – model is deleted and gc runs after every run
    del model
    gc.collect()

print_results(results_smell1, 'Smell 1 – Unbounded Graph Expansion')

---
## Section 2 – SMELL 2: Graph-Constant Bottleneck

**Active smell:** The entire training dataset is embedded as `tf.constant` tensors directly inside the TensorFlow computational graph. These tensors are permanently resident in memory and cannot be garbage-collected for the lifetime of the session.

**Absent smells:** `K.clear_session()` is called between runs (no graph expansion); the model is deleted after each run (no GPU leak); data is used in its original float32 form with no intermediate float64 copy (no shape mismatch).

In [ ]:
# ============================================================
# SMELL 2 – Graph-Constant Bottleneck (isolated)
# ============================================================
os.makedirs('emissions', exist_ok=True)
results_smell2 = []

# SMELL 2 ACTIVE – Entire training dataset embedded as tf.constant.
# These tensors are permanently pinned inside the TF graph and cannot
# be freed by the garbage collector for the lifetime of the session.
X_train_const     = tf.constant(X_train,     dtype=tf.float32)  # whole dataset in graph
y_cat_train_const = tf.constant(y_cat_train, dtype=tf.float32)

for run in range(1, 11):
    print(f"\n{'='*60}")
    print(f'  [Smell 2] Run {run}/10')
    print(f"{'='*60}")

    # No graph expansion smell – session is cleared between runs
    K.clear_session()

    tracker = EmissionsTracker(
        project_name=f'CNN_SVHN_Smell2_Run{run}',
        output_dir='./emissions',
        save_to_file=True,
        log_level='warning'
    )
    tracker.start()

    model      = build_cnn()
    early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

    # No shape mismatch smell – use original float32 arrays directly
    history = model.fit(
        X_train, y_cat_train,
        epochs=50, batch_size=32,
        validation_data=(X_test, y_cat_test),
        callbacks=[early_stop],
        verbose=1
    )

    co2_kg     = tracker.stop()
    energy_kwh = tracker.final_emissions_data.energy_consumed

    epoch_stopped = len(history.history['loss'])
    eval_res      = model.evaluate(X_test, y_cat_test, verbose=0)

    results_smell2.append(dict(
        run=run,
        epoch_stopped=epoch_stopped,
        test_loss=eval_res[0],
        test_accuracy=eval_res[1],
        test_precision=eval_res[2],
        test_recall=eval_res[3],
        energy_kwh=energy_kwh,
        co2_kg=co2_kg
    ))
    print(f'Run {run} | Epoch stopped: {epoch_stopped} | Acc: {eval_res[1]:.4f} | '
          f'Energy: {energy_kwh:.6f} kWh | CO2: {co2_kg:.6f} kg')

    # No GPU memory smell – model is deleted and gc runs after every run
    del model
    gc.collect()

print_results(results_smell2, 'Smell 2 – Graph-Constant Bottleneck')

---
## Section 3 – SMELL 3: GPU Released Memory Failure

**Active smell:** After each run the model object is **not** deleted and `K.clear_session()` is **not** called, so GPU memory allocated for model weights and optimizer states is never explicitly freed. Only `gc.collect()` is called, which reclaims CPU memory only — GPU memory remains occupied and accumulates across runs.

**Absent smells:** `K.clear_session()` is still called at the top of each loop iteration before building the new model (preventing graph expansion); no graph-constant tensors are created; data is used in its original float32 form (no shape mismatch).

In [ ]:
# ============================================================
# SMELL 3 – GPU Released Memory Failure (isolated)
# ============================================================
os.makedirs('emissions', exist_ok=True)
results_smell3 = []

for run in range(1, 11):
    print(f"\n{'='*60}")
    print(f'  [Smell 3] Run {run}/10')
    print(f"{'='*60}")

    # No graph expansion smell – session is cleared between runs
    K.clear_session()

    tracker = EmissionsTracker(
        project_name=f'CNN_SVHN_Smell3_Run{run}',
        output_dir='./emissions',
        save_to_file=True,
        log_level='warning'
    )
    tracker.start()

    model      = build_cnn()
    early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

    # No shape mismatch smell – use original float32 arrays directly
    history = model.fit(
        X_train, y_cat_train,
        epochs=50, batch_size=32,
        validation_data=(X_test, y_cat_test),
        callbacks=[early_stop],
        verbose=1
    )

    co2_kg     = tracker.stop()
    energy_kwh = tracker.final_emissions_data.energy_consumed

    epoch_stopped = len(history.history['loss'])
    eval_res      = model.evaluate(X_test, y_cat_test, verbose=0)

    results_smell3.append(dict(
        run=run,
        epoch_stopped=epoch_stopped,
        test_loss=eval_res[0],
        test_accuracy=eval_res[1],
        test_precision=eval_res[2],
        test_recall=eval_res[3],
        energy_kwh=energy_kwh,
        co2_kg=co2_kg
    ))
    print(f'Run {run} | Epoch stopped: {epoch_stopped} | Acc: {eval_res[1]:.4f} | '
          f'Energy: {energy_kwh:.6f} kWh | CO2: {co2_kg:.6f} kg')

    # SMELL 3 ACTIVE – model is NOT deleted and K.clear_session() is NOT called here.
    # GPU memory for weights and optimizer states is never explicitly freed.
    # gc.collect() only reclaims CPU memory; GPU memory remains occupied.
    gc.collect()   # CPU memory only – GPU memory stays allocated

In [ ]:
print_results(results_smell3, 'Smell 3 – GPU Released Memory Failure')

---
## Section 4 – SMELL 4: Shape Mismatch Leak

**Active smell:** The float32 training data is first upcast to float64, and then cast back to float32. Both copies — the intermediate float64 array (double the memory) and the redundant float32 copy — remain simultaneously allocated in RAM throughout all 10 runs, wasting memory.

**Absent smells:** `K.clear_session()` is called between runs (no graph expansion); no graph-constant tensors are created; the model is deleted and `gc.collect()` is called after every run (no GPU leak).

In [ ]:
# ============================================================
# SMELL 4 – Shape Mismatch Leak (isolated)
# ============================================================
os.makedirs('emissions', exist_ok=True)
results_smell4 = []

# SMELL 4 ACTIVE – Unnecessary float64 upcast doubles memory footprint.
# The float64 arrays and the redundant float32 copies both remain
# allocated in RAM simultaneously for the entire experiment.
X_train_f64 = X_train.astype(np.float64)      # unnecessary upcast – doubles RAM usage
X_test_f64  = X_test.astype(np.float64)
X_train_f32 = X_train_f64.astype(np.float32)  # redundant float32 copy
X_test_f32  = X_test_f64.astype(np.float32)
# X_train_f64 and X_train_f32 now live in RAM simultaneously

for run in range(1, 11):
    print(f"\n{'='*60}")
    print(f'  [Smell 4] Run {run}/10')
    print(f"{'='*60}")

    # No graph expansion smell – session is cleared between runs
    K.clear_session()

    tracker = EmissionsTracker(
        project_name=f'CNN_SVHN_Smell4_Run{run}',
        output_dir='./emissions',
        save_to_file=True,
        log_level='warning'
    )
    tracker.start()

    model      = build_cnn()
    early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

    # Training uses the redundant float32 copy produced via the float64 upcast
    history = model.fit(
        X_train_f32, y_cat_train,
        epochs=50, batch_size=32,
        validation_data=(X_test_f32, y_cat_test),
        callbacks=[early_stop],
        verbose=1
    )

    co2_kg     = tracker.stop()
    energy_kwh = tracker.final_emissions_data.energy_consumed

    epoch_stopped = len(history.history['loss'])
    eval_res      = model.evaluate(X_test_f32, y_cat_test, verbose=0)

    results_smell4.append(dict(
        run=run,
        epoch_stopped=epoch_stopped,
        test_loss=eval_res[0],
        test_accuracy=eval_res[1],
        test_precision=eval_res[2],
        test_recall=eval_res[3],
        energy_kwh=energy_kwh,
        co2_kg=co2_kg
    ))
    print(f'Run {run} | Epoch stopped: {epoch_stopped} | Acc: {eval_res[1]:.4f} | '
          f'Energy: {energy_kwh:.6f} kWh | CO2: {co2_kg:.6f} kg')

    # No GPU memory smell – model is deleted and gc runs after every run
    del model
    gc.collect()

print_results(results_smell4, 'Smell 4 – Shape Mismatch Leak')